# Direct Orthogonal Feature Correction 

This approach operates on the internal mathematics of the drone's vision system, aiming to align how a frozen vision foundation model (VFM) represents the wolrd across different flight heights

## Orthogonal Procrustes

Orthogonal Procrustes mapping acts as a high-dimensional spatial alignment tool. It seeks to find a single optimal rotation and reflection matrix that mathematically rotates the 80m feature cluster so that it aligns directly with the 30m cluster. This allows a simple classifier (a linear probe) trained exclusively on 30m features to work seamlessly on 80m data

### Mathematics of Orthogonal Procrustes

The goal is to map the features of a high-altitude matrix ($X_{80}$) into a low-altitude reference space ($X_{30}$) using an orthogonal matrix $R^*$. We solve for the optimal orthogonal mapping matrix $R^*$ that minimizes the sum of squared Euclidean distances (the Frobenius norm) between the mapped and target features
$$R^* = \arg\min_{R^T R = I} \|X_{80} R - X_{30}\|_F^2$$
Where $R^T R = I$ enforces the orthogonality constraint. This constraint is vital because orthogonal transformations preserve vector lengths, angles, and pairwise Euclidean distances.They represent a pure, rigid rotation/ reflection of the feature space without distorting the underlying data structure

**The SVD solution**

According to Schönemann’s generalized solution, this constrained optimization problem has an elegant closed-form solution using Singular Value Decomposition (SVD)
1. First, compute the cross-covariance matrix of your paired calibration sets: $X_{80}^T X_{30}$
2. Calculate its SVD: $$X_{80}^T X_{30} = U \Sigma V^T$$
3. The optimal orthogonal rotation matrix $R^*$ is constructed by multiplying the left and right singular vectors $$R^* = U V^T$$

**Strict vs. Centred Mapping**

two variations of this mathematical transformation:
1. Strict Orthogonal Mapping (No Translation): The high-altitude features are rotated directly using the matrix $$\hat{z}_{30} = z_{80} R^*$$
2. Centred Orthogonal Mapping (With Translation): A common Procrustes workflow centers each domain around its mean vector before rotating, and then translates them into the target space $$\hat{z}_{30} = (z_{80} - \mu_{80}) R^* + \mu_{30}$$

where $\mu_{80}$ and $\mu_{30}$ are the average feature vectors. Comparing these two variants helps establish whether the altitude shift is a pure high-dimensional rotation or if it involves a significant mean shift (translation)

### Setting Up the Comparison Groups (The 7 Study "Arms")

To rigorously evaluate whether an orthogonal transformation is the correct mathematical model for altitude shifts, this framework structures the feature experiment into seven treatment arms, evaluated on the same held-out physical objects: 
- **Arm A (Source-Domain Reference)**: Fixed 30m linear probe evaluated on real 30m features *Establishes upper-bound reference accuracy*
- **Arm B (Uncorrected Cross-Altitude)**: The 30m probe evaluated directly on raw, uncorrected real 80m features. * Establishes the cross-altitude performance drop*
- **Arm C (Strict Procrustes)**:  Real 80m features mapped by the strict orthogonal matrix ($R^* = U V^T$), then passed to the unchanged 30m probe. *Directly tests the strict orthogonal correction hypothesis*
- **Arm D (Centred Procrustes)**: Real 80m features mapped using the translation inclusive centred method, then passed to the 30m probe. *Tests is a translation (mean shift) is required alongside rotation*
- **Arm E (Mean shift only)**: Adjusting 80m features solely by subtracting $\mu_{80}$ and adding $\mu_{30}$ without any rotation. *Tests if the benefit comes purely from mean alignment*
- **Arm F (Unconstrained Linear Mapping)**: Utilising a regularised linear map (like Ridge regression) trained on the calibration pairs, omitting the orthognality constraint. *Tests if forcing orthogonality is a useful restriction or if it over-constrains the transformation*
- **Arm G (Negative Control)**: Running the Procrustes calculation after randomly shuffling the pairing identities within the calibrated data. *Proves that the mapping relies on precise, 1-to-1 physical object matching rather than generic domain statistics*

# Frozen vision encoders

This is the backbone of the Feature-Level correction approach. This approach focusses on how modern, pretrained Vision Foundation Models (VFMs) - such as DINOv3, CLIP, or SAM - internally represent visual scale shifts.